In [1]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from src.features import build_sessions, attach_labels, count_vector, apply_tfidf

In [2]:
df = pd.read_csv('../data/parsed/hdfs_parsed.csv')
print("Rows:", len(df))
df.head()

Rows: 200000


,date,time,pid,level,component,content,BlockId,ts,EventId,EventTemplate
0,81109,203518,143,INFO,dfs.DataNode$DataXceiver,Receiving block blk_-1608999687919862906 src: ...,blk_-1608999687919862906,2008-11-09 20:35:18,1,Receiving block <BLK> src: /<IP> dest: /<IP>
1,81109,203518,35,INFO,dfs.FSNamesystem,BLOCK* NameSystem.allocateBlock: /mnt/hadoop/m...,blk_-1608999687919862906,2008-11-09 20:35:18,2,BLOCK* NameSystem.allocateBlock: <PATH> <BLK>
2,81109,203519,143,INFO,dfs.DataNode$DataXceiver,Receiving block blk_-1608999687919862906 src: ...,blk_-1608999687919862906,2008-11-09 20:35:19,1,Receiving block <BLK> src: /<IP> dest: /<IP>
3,81109,203519,145,INFO,dfs.DataNode$DataXceiver,Receiving block blk_-1608999687919862906 src: ...,blk_-1608999687919862906,2008-11-09 20:35:19,1,Receiving block <BLK> src: /<IP> dest: /<IP>
4,81109,203519,145,INFO,dfs.DataNode$PacketResponder,PacketResponder 1 for block blk_-1608999687919...,blk_-1608999687919862906,2008-11-09 20:35:19,3,PacketResponder <NUM> for block <BLK> terminating


In [3]:
sessions = build_sessions(df)
sessions = attach_labels(sessions, '../data/raw/anomaly_label.csv')

print("Sessions:", len(sessions))
print("Anomalies:", sessions['y'].sum(), f"({sessions['y'].mean()*100:.2f}%)")
sessions.head()

Sessions: 15639
Anomalies: 698 (4.46%)


,BlockId,EventSequence,y
0,blk_-1001553972418305662,"[2, 1, 1, 1, 5, 5, 3, 4, 3, 4, 5, 3, 4]",0
1,blk_-1001893162880667454,"[1, 2, 1, 1, 3, 4, 3, 4, 3, 4, 5, 5, 5]",0
2,blk_-1007760638588980360,"[1, 2, 1, 1, 3, 4, 5, 3, 4, 3, 4, 5, 5]",0
3,blk_-1008202429000899029,"[2, 1, 1, 1, 3, 4, 3, 4, 3, 4, 5, 5, 5]",0
4,blk_-1010952805175971965,"[2, 1, 1, 1, 5, 5, 5, 3, 4, 3, 4, 3, 4]",0


In [4]:
print("--- NORMAL ---")
for seq in sessions[sessions.y == 0]['EventSequence'].head(3):
    print(seq)

print("\n--- ANOMALY ---")
for seq in sessions[sessions.y == 1]['EventSequence'].head(3):
    print(seq)

--- NORMAL ---
[2, 1, 1, 1, 5, 5, 3, 4, 3, 4, 5, 3, 4]
[1, 2, 1, 1, 3, 4, 3, 4, 3, 4, 5, 5, 5]
[1, 2, 1, 1, 3, 4, 5, 3, 4, 3, 4, 5, 5]

--- ANOMALY ---
[1, 1, 1, 2, 3, 4, 3, 4, 3, 4, 5, 5, 5]
[2, 1, 1, 16]
[2, 1]


In [5]:
X_counts, event_names = count_vector(sessions)
X = apply_tfidf(X_counts)
y = sessions['y'].values

print("Matrix shape:", X.shape)
print("Events:", event_names)

Matrix shape: (15639, 25)
Events: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25]


In [6]:
np.save('../data/features/X.npy', X)
np.save('../data/features/y.npy', y)
sessions.to_csv('../data/features/sessions.csv', index=False)
print("Saved")

Saved
